# MIPROv2 Prompt Optimizer — Diagnóstico de incidencias técnicas

**Caso de uso:** un bot de soporte técnico recibe errores e incidencias reportadas por usuarios y debe responder con un diagnóstico estructurado de 4 secciones: **Diagnóstico**, **Causa probable**, **Solución inmediata** (pasos numerados) y **Seguimiento**.

Con el seed prompt `"Eres un agente de soporte técnico."`, el bot responde en prosa sin estructura — el LLM judge lo penaliza porque no sigue el formato esperado. MIPROv2 encuentra la combinación de instrucción + ejemplos few-shot que enseña el formato exacto.

El dataset cubre 4 tipos de incidencias (notificaciones, rendimiento, exportaciones, permisos) con 2 casos cada uno. Cada `Dataset` tiene su propio `context` con la documentación técnica relevante.

**Por qué MIPROv2 y no GEPA:** el formato de 4 secciones es difícil de especificar solo con instrucciones — los ejemplos few-shot lo transmiten mucho más efectivamente.

# Instalation

In [ ]:
!pip install alquimia-fair-forge[prompt-optimizer] langchain-groq

## Setup

In [1]:
import os
import logging
import getpass

logging.getLogger("httpx").setLevel(logging.WARNING)

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parents[3]))
sys.path.insert(0, str(Path().resolve().parents[3] / "examples"))

## 1. Retriever

Carga los casos de incidencias y sus diagnósticos esperados. Cada `Dataset` incluye la documentación técnica relevante en `context`.

In [3]:
import json
from pathlib import Path
from fair_forge import Retriever
from fair_forge.schemas import Dataset

class IncidentsRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        data_path = Path("../data/dataset.json")
        with open(data_path, encoding="utf-8") as f:
            return [Dataset.model_validate(item) for item in json.load(f)]

c:\Users\ericf\Desktop\Alquimia\fair-forge\.venv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 2. Model

El modelo que MIPROv2 usa para generar variantes de instrucción. También actúa como agente bajo optimización (ejecuta cada prompt candidato contra el dataset).

In [4]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")

## 3. Run the optimizer

- `seed_prompt`: el prompt actual del agente — deliberadamente vago.
- `objective`: describe qué hace un buen bot de onboarding. MIPROv2 lo usa para generar variantes de instrucción y evaluar respuestas.
- `num_candidates`: cuántas variantes de instrucción generar (default: 10).
- `num_trials`: cuántas combinaciones `(instrucción, ejemplos)` explorar con Bayesian Optimization (default: 20).

In [5]:
from fair_forge.prompt_optimizer.mipro import MIPROv2Optimizer

result = MIPROv2Optimizer.run(
    retriever=IncidentsRetriever,
    model=model,
    seed_prompt="Eres un agente de soporte técnico.",
    objective=(
        "Responder incidencias técnicas con un diagnóstico estructurado en exactamente 4 secciones: "
        "1) Diagnóstico: qué está pasando en una línea. "
        "2) Causa probable: por qué ocurre en una línea. "
        "3) Solución inmediata: pasos numerados para resolver el problema. "
        "4) Seguimiento: próximos pasos o medida de prevención en una línea. "
        "Usar únicamente la información del contexto proporcionado. No agregar secciones adicionales."
    ),
    num_candidates=10,
    num_trials=20,
)

Evaluating seed prompt on 8 examples...
Generating 10 instruction candidates...


Evaluating trials:   0%|          | 0/20 [00:00<?, ?it/s]

## 4. Trial history

Cada trial explora una combinación de instrucción y set de ejemplos. Un `★` indica un nuevo mejor score.

In [6]:
best_so_far = 0.0
for t in result.trials:
    is_best = t.score > best_so_far
    if is_best:
        best_so_far = t.score
    marker = " ★ new best" if is_best else ""
    print(f"  Trial {t.trial + 1:2d} — instruction #{t.instruction_idx + 1}, demo set #{t.demo_set_idx + 1} — score: {t.score:.2f}{marker}")

  Trial  1 — instruction #4, demo set #5 — score: 0.82 ★ new best
  Trial  2 — instruction #8, demo set #3 — score: 0.75
  Trial  3 — instruction #2, demo set #1 — score: 0.80
  Trial  4 — instruction #1, demo set #5 — score: 0.86 ★ new best
  Trial  5 — instruction #7, demo set #4 — score: 0.78
  Trial  6 — instruction #1, demo set #5 — score: 0.84
  Trial  7 — instruction #9, demo set #2 — score: 0.78
  Trial  8 — instruction #2, demo set #1 — score: 0.75
  Trial  9 — instruction #4, demo set #3 — score: 0.88 ★ new best
  Trial 10 — instruction #5, demo set #2 — score: 0.81
  Trial 11 — instruction #4, demo set #3 — score: 0.85
  Trial 12 — instruction #3, demo set #4 — score: 0.83
  Trial 13 — instruction #6, demo set #4 — score: 0.78
  Trial 14 — instruction #1, demo set #2 — score: 0.78
  Trial 15 — instruction #10, demo set #4 — score: 0.80
  Trial 16 — instruction #3, demo set #5 — score: 0.73
  Trial 17 — instruction #5, demo set #3 — score: 0.89 ★ new best
  Trial 18 — instruc

## 5. Results

In [7]:
print(f"Initial score : {result.initial_score:.2f} / 1.00  (LLM judge · {result.n_examples} examples)")
print(f"Final score   : {result.final_score:.2f} / 1.00  (LLM judge · {result.n_examples} examples)")
print(f"Trials run    : {result.trials_run}")
print()
print("=== Seed prompt ===")
print("Eres un agente de soporte técnico.")
print()
print("=== Optimized instruction ===")
print(result.optimized_instruction)
print()
print("=== Selected examples ===")
for demo in result.demos:
    print(f"User: {demo.query}")
    print(f"Assistant: {demo.response}")
    print()

Initial score : 0.20 / 1.00  (LLM judge · 8 examples)
Final score   : 0.89 / 1.00  (LLM judge · 8 examples)
Trials run    : 20

=== Seed prompt ===
Eres un agente de soporte técnico.

=== Optimized instruction ===
Actúa como un agente de soporte técnico conciso. Responde a incidencias técnicas con un diagnóstico estructurado en 4 secciones: Diagnóstico, Causa probable, Solución inmediata y Seguimiento, utilizando el mínimo de palabras necesarias para ser claro.

=== Selected examples ===
User: La vista de cronograma tarda más de 30 segundos en cargar. El proyecto tiene 800 tareas.
Assistant: **Diagnóstico:** La vista de cronograma está cargando las 800 tareas completas en memoria, lo que supera el umbral de rendimiento recomendado.

**Causa probable:** La paginación de cronograma no está activada, forzando la carga completa de todos los datos.

**Solución inmediata:**
1. Andá a Configuración del proyecto > Vista > Paginación de cronograma.
2. Activá la opción y configurá el tamaño de p